<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/2_feature_engineering/2_3_ic_score_and_correlation_features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2_3_correlation_features

## Resumen


Esta notebook se centra en el **cálculo y análisis de indicadores técnicos intradía** aplicados al índice  **Micro E-mini Nasdaq 100 (MNQ)**, con el fin de construir features que sirvan como insumo para modelos predictivos.

## 0. Configuración del Entorno


### 0.1. Clonado de repositorio / Acceso a Drive

In [1]:
#Clonamos el repo
#LINK DE REPOSITORIO: https://github.com/GUNAPILLCO/neural_profit
!git clone https://github.com/GUNAPILLCO/neural_profit.git

Cloning into 'neural_profit'...
remote: Enumerating objects: 494, done.
remote: Counting objects: 100% (114/114), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 494 (delta 69), reused 31 (delta 21), pack-reused 380 (from 2)
Receiving objects: 100% (494/494), 250.22 MiB | 25.74 MiB/s, done.
Resolving deltas: 100% (284/284), done.
Updating files: 100% (60/60), done.


In [2]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Mounted at /content/drive


### 0.2. Instalación de librerías


In [11]:
!{sys.executable} -m pip install -q ta
print("Librería instalada: technical-analysis")

  Preparing metadata (setup.py) ... done
Librería instalada: technical-analysis


### 0.3. Importación de librerías


In [12]:
import sys
import re
#Instalación de librería pandas_market_calendars
#!{sys.executable} -m pip install -q pandas_market_calendars
#print("Librería instalada: pandas_market_calendars")


from functools import reduce
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import requests
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos
import ta
import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
# Calendario de mercados
#import pandas_market_calendars as mcal

from ta.momentum import StochasticOscillator, ROCIndicator
from ta.volatility import BollingerBands, AverageTrueRange

from scipy.stats import spearmanr


### 0.4. Carga de datasets



In [13]:
def load_data(data: str):

    data_path = f'{drive_path}/2_feature_engineering/mnq_{data}.parquet'

    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)

    # Asegurar que el índice esté en formato datetime
    df.index = pd.to_datetime(df.index)

    # Crear una nueva columna 'date' con la fecha extraída del índice
    df['date'] = df.index.date

    # Reordenar columnas: 'date', 'time_str', y luego el resto
    cols = ['date'] + [col for col in df.columns if col not in ['date']]

    df = df[cols]

    return df

In [14]:
def load_ic(data: str):

    data_path = f'{drive_path}/2_feature_engineering/ic_{data}.parquet'

    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)

    return df

In [15]:
mnq_technical_indicators = load_data('technical_indicators')
ic_technical_indicators = load_ic('technical_indicators')

In [16]:
mnq_alpha_factors = load_data('alpha_factors')
ic_alpha_factors = load_ic ('alpha_factors')

In [20]:
ic_technical_indicators.columns

Index(['indicador', 'ic_mean_30min', 'ic_std_30min', 'ic_mean_60min',
       'ic_std_60min', 'ic_mean_90min', 'ic_std_90min'],
      dtype='object')

In [19]:
ic_alpha_factors.columns

Index(['indicador', 'ic_mean_30min', 'ic_std_30min', 'ic_mean_60min',
       'ic_std_60min', 'ic_mean_90min', 'ic_std_90min'],
      dtype='object')

### 0.5. Info de dataset MNQ


In [21]:
def info_dataset(df):
  print("Información del dataset:\n")

  # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  print(f"\tCantidad de días: {num_dias}")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['close']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  print(f"\tRegistros por día: {int(promedio_por_fecha)}")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo


  print(f"\tHora diaria de inicio {primer_hora}")
  print(f"\tHora diaria de final {ultima_hora}")
  print(f"\tZona horaria: {zona_horaria}")

In [22]:
info_dataset(mnq_alpha_factors)

Información del dataset:

	Cantidad de días: 1311
	Registros por día: 481
	Hora diaria de inicio 08:00
	Hora diaria de final 16:00
	Zona horaria: America/New_York


In [23]:
info_dataset(mnq_technical_indicators)

Información del dataset:

	Cantidad de días: 1311
	Registros por día: 481
	Hora diaria de inicio 08:00
	Hora diaria de final 16:00
	Zona horaria: America/New_York


## 1. Alineación de datasets

Empezaremos alineando horizontalmente los datasets `mnq_technical_indicators` y `mnq_alpha_factors`.

In [24]:
# Alineación horizontal por índice (datetime)
mnq_features_combined = pd.concat([mnq_technical_indicators, mnq_alpha_factors], axis=1)
# Eliminar columnas duplicadas (conservando la primera aparición)
mnq_features_combined = mnq_features_combined.loc[:, ~mnq_features_combined.columns.duplicated()]
# Confirmar que los índices estén alineados
mnq_features_combined = mnq_features_combined.sort_index()

In [27]:
mnq_features_combined.columns

Index(['date', 'open', 'high', 'low', 'close', 'volume', 'target_return_30',
       'target_return_60', 'target_return_90', 'rsi_14', 'rsi_7',
       'momentum_10', 'momentum_5', 'macd', 'price_ema20', 'price_ema30',
       'stoch_k_20', 'stoch_k_30', 'bb_20', 'bb_30', 'bb_60', 'atr_norm',
       'roc_20', 'roc_30', 'roc_60', 'rev_mom_z_30', 'rev_mom_z_45',
       'rev_mom_z_60', 'rev_mom_z_90', 'rev_score_30', 'rev_score_45',
       'rev_score_60', 'rev_score_90', 'rev_mom_vol_z_45', 'rev_mom_vol_z_60',
       'rev_mom_vol_z_90', 'ire_60', 'ire_90'],
      dtype='object')

Luego alianeamos verticalmente los datasets `ic_technical_indicators` y `ic_alpha_factors`.


In [42]:
ic_features_combined = pd.concat([ic_technical_indicators, ic_alpha_factors], axis=0)

In [43]:
ic_features_combined = ic_features_combined.rename(columns={"indicador": "feature"})


In [44]:
ic_features_combined

,feature,ic_mean_30min,ic_std_30min,ic_mean_60min,ic_std_60min,ic_mean_90min,ic_std_90min
0,rsi_14,-0.088180,0.172203,-0.132798,0.200280,-0.176524,0.202798
1,rsi_7,-0.062888,0.137946,-0.095645,0.152071,-0.127395,0.152254
0,momentum_10,-0.053240,0.129744,-0.082424,0.136893,-0.112294,0.134865
1,momentum_5,-0.040213,0.093976,-0.060545,0.097287,-0.082049,0.096481
0,macd,-0.018010,0.135153,-0.026394,0.122985,-0.035572,0.121715
1,price_ema20,-0.072318,0.158994,-0.111052,0.177833,-0.151308,0.174142
2,price_ema30,-0.087274,0.176637,-0.132857,0.203450,-0.180860,0.199570
1,stoch_k_20,-0.070241,0.147213,-0.105944,0.159769,-0.141962,0.160943
2,stoch_k_30,-0.087019,0.170373,-0.132252,0.192868,-0.175458,0.193385
1,bb_20,-0.063539,0.141046,-0.097252,0.152624,-0.130950,0.154313


## 2. Criterio IC Score


Nuestro enfoque para la selección de los mejores features es el **IC Score**

El IC score combina dos métricas clave para evaluar la calidad de un indicador técnicos:

- |IC medio|: mide el poder predictivo promedio del indicador (cuánto se asocia su valor con el retorno futuro).

- IC std: mide la volatilidad o inestabilidad del indicador a lo largo del tiempo (días en tu caso).

El cociente `|IC| / std` representa un signal-to-noise ratio: cuánta señal útil aporta el indicador, ajustada por su variabilidad diaria.

Esto nos permite seleccionar features que no solo tienen buen rendimiento promedio, sino que además son consistentes. En otras palabras, es una forma de medir la **relación señal/ruido** de cada feature.

**Ventajas del IC Score como criterio de selección**

| Criterio | Descripción |
| --- | --- |
| **Objetividad** | Es una métrica cuantitativa clara, sin intervención manual ni subjetiva. |
| **Consistencia** | Premia indicadores que funcionan bien y consistentemente a lo largo de días. |
| **Robustez estadística** | Penaliza indicadores que parecen buenos pero son muy inestables (overfitting). |
| **Comparabilidad** | Permite ordenar decenas de indicadores heterogéneos en una sola escala. |
| **Modelado automatizado** | Compatible con pipelines de selección automática, sin intervención humana. |


**¿Por qué es mejor que usar solo el IC medio?**

Porque un factor puede tener un IC alto pero extremadamente volátil, lo cual indica que su utilidad es espuria o inconstante. En cambio, un factor con IC moderado pero muy estable suele ser más confiable para modelos reales.

En el trading cuantitativo institucional, es común que los equipos usen IC Score o variantes del Information Ratio (IR) como filtro previo antes de usar un factor en producción.

Ejemplo concreto:

- `atr_norm`: tiene el IC medio más alto (+0.0899), pero su std también es muy alta (0.2389) → bajo IC score ≈ 0.38

- `rsi_14`: IC medio menor (-0.0882), pero más estable (std = 0.17) → mejor IC score ≈ 0.51

Entonces, si tenés que elegir entre ellos para un modelo robusto, el IC score sugiere que `rsi_14` es más confiable, aunque su IC medio sea menor.

**Conclusión**

El IC score es una métrica más equilibrada, porque considera tanto la fuerza como la confiabilidad del factor.

Usarlo como criterio principal me protege de elegir factores ruidosos, y mejora la calidad de las señales que entrarán en mi modelo, especialmente si:

- Estoy evaluando muchos indicadores (como es nuestro caso).

- Voy a usar modelos complejos que son sensibles al ruido (como redes neuronales).

- En una etapa exploratoria podríamos automatizar y escalar el proceso de feature selection.

### 2.1. Calculo de IC Score para las ventanas de 30, 60 y 90 minutos.

Calcular la nueva columna IC_score para cada ventana de predicción

In [46]:
targets = ['30','60','90']
for target in targets:
  ic_features_combined[f"ic_score_{target}min"] =ic_features_combined[f"ic_mean_{target}min"].abs() / ic_features_combined[f"ic_std_{target}min"]


Con el siguiente código se generan tres datasets distintos, cada uno ordenado por el valor de IC_score correspondiente a su horizonte de predicción (30, 60 y 90 minutos).  De esta manera, es posible identificar qué indicadores técnicos tienen mayor peso relativo en cada ventana temporal.

In [48]:
ic_features_30 = ic_features_combined.sort_values(
    by="ic_score_30min", ascending=False
).reset_index(drop=True)

ic_features_60 = ic_features_combined.sort_values(
    by="ic_score_60min", ascending=False
).reset_index(drop=True)

ic_features_90 = ic_features_combined.sort_values(
    by="ic_score_90min", ascending=False
).reset_index(drop=True)

El siguiente código construye un nuevo dataset llamado `ic_features_comparacion`, donde se alinean horizontalmente las columnas `feature` de cada uno de los tres rankings generados previamente (`ic_features_30`, `ic_features_60` e `ic_features_90`).  

Cada columna contiene la lista ordenada de indicadores técnicos según su ic_score en el horizonte correspondiente (30, 60 y 90 minutos), lo que nos permite comparar fácilmente cuáles factores se mantienen, suben o bajan en importancia entre nuestras diferentes ventanas de predicción.

In [49]:
ic_features_comparison = pd.DataFrame({
    "features_to_30min": ic_features_30["feature"].reset_index(drop=True),
    "features_to_60min": ic_features_60["feature"].reset_index(drop=True),
    "features_to_90min": ic_features_90["feature"].reset_index(drop=True),
})

### 2.2. Resultados

In [50]:
#Resultados basados en IC_score
ic_features_comparison

,features_to_30min,features_to_60min,features_to_90min
0,ire_90,ire_60,ire_60
1,ire_60,ire_90,ire_90
2,rev_mom_z_90,rev_mom_z_90,rev_mom_z_90
3,roc_60,roc_60,roc_60
4,rev_mom_z_60,rev_mom_z_60,rev_mom_z_60
5,rev_mom_vol_z_90,bb_60,bb_60
6,rev_score_90,rev_score_90,stoch_k_30
7,rev_mom_vol_z_60,rev_mom_vol_z_90,price_ema30
8,bb_60,rev_mom_z_45,rev_mom_z_45
9,rev_mom_z_45,stoch_k_30,rev_score_90


### 2.3. Análisis de factores por horizonte de predicción


1. Factores dominantes y estables

    - `ire_60` e `ire_90`  lideran claramente en todos los horizontes, con `ic_score` muy altos (>2). Son los factores más robustos para explicar retornos, independientemente de la ventana.  
    - `rev_mom_z_90` y `roc_60` se mantienen siempre en el top 5 de importancia. Tienen estabilidad y buena capacidad predictiva.  
    - `atr_norm` y `macd` aunque aparecen al final de los rankings, son consistentes: siempre están incluidos, lo que indica una señal débil pero persistente.  

2. Factores con estabilidad media

    - `bb_60` fuerte en 60 y 90 minutos, algo más abajo en 30 min. Su importancia crece con el horizonte.  
    - `stoch_k_30`, `stoch_k_20`, `price_ema30`, `price_ema20`, `rsi_14`, `rsi_7` siempre están presentes en posiciones intermedias, mostrando que aportan valor, aunque no son dominantes.  
    - `rev_score_60` y `rev_score_90` mantienen relevancia en los tres horizontes.  

3. Factores que ganan importancia en horizontes largos

    - `rev_mom_vol_z_90` es fuerte en 30 min (#5) y 60 min (#7), pero pierde peso en 90 min (#26). Podría ser un factor de corto/medio plazo.  
    - `rev_mom_z_45` y `rev_mom_z_60` su ranking mejora en 60 y 90 min, sugiriendo que capturan mejor reversión/momentum en ventanas largas.  
    - `rev_mom_z_90` claramente el más fuerte entre los reversales a plazos largos.  

4. Factores que tienden a rotar o perder fuerza

    - `momentum_5` y `momentum_10` son bajos en todos los horizontes, aunque siempre aparecen. Aportan señal, pero débil.  
    - `roc_20`, `roc_30` están mejor posicionados en 30 min, luego decaen en 90 min. Son más útiles en ventanas cortas.  
    - `rev_mom_z_30`  pierde relevancia al aumentar la ventana.  

5. Patrones generales

    - Los feature de reversión (`rev_mom_z`, `rev_score`, `rev_mom_vol_z`) ganan fuerza a medida que el horizonte crece. Ejemplo: `rev_mom_z_90` es top-3 siempre.  
    - Momentum clásico (`roc`, `momentum`, `ema`, `rsi`, `stoch`, `bb`) aportan pero se ven desplazados en horizontes largos por factores de reversión.  
    - Volatilidad (`atr_norm`) no es dominante, pero constante: puede ser útil como complemento en modelos, aunque no como feature principal.  

### 2.4. Conclusión basada en análisis de ic_score

- Factores robustos y no negociables: `ire_60`, `ire_90`, `rev_mom_z_90`, `roc_60`, `bb_60`.  
- Factores secundarios que añaden diversificación: `stoch_k_30`, `stoch_k_20`, `price_ema30`, `rsi_14`, `rev_score_90`.  
- Factores complementarios: `atr_norm`, `macd`, `momentum_5`, `momentum_10`.  

### 3. Correlación

Ya ordenamos los indicadores por IC Score, que es un métrica robusta de señal ajustada por estabilidad. Pero necesitamos controlar la redundancia (multicolinealidad). Ya que indicadores con alta correlación podrían aportar información duplicada, y:

- Distorsionar modelos lineales
- Inflar varianzas en los coeficientes
- Hacer innecesariamente complejo el modelo

**¿Cómo lo controlo?**
<br>
Usaré una matriz de correlación entre indicadores, típicamente sobre sus valores normalizados (z-score por día) para detectar grupos redundantes.


#### 3.1. Construcción de dataset de features

Recordemos que tenemos un listado con los features a conservar, filtramos el dataset para que solo queden las columnas correspondientes a esos features:

In [53]:
columnas_base = ['date', 'open', 'high', 'low','close','volume']
columnas_target = ['target_return_30','target_return_60','target_return_90']

# Unimos las listas de columnas a eliminar
cols_drop = columnas_base + columnas_target

# Eliminamos esas columnas
mnq_features = mnq_features_combined.drop(columns=cols_drop, errors="ignore")

### 3.2. Calculo de correlación

Calculamos la matriz de correlación:

In [70]:
mnq_features_corr = mnq_features.corr()

### 3.3. Buscamos la mayor correlación de cada feature

Teniendo:
- ic_features_combined: dataset con columna 'feature'
- mnq_features: dataset con todos los features calculados
- mnq_features_corr : matriz de correlación

Vamos a calcular los 3 factores más correlacionados con cada factor:

In [56]:
# Para cada factor en Iic_features_combined, encontrar los 5 más correlacionados
top5_features = []
top5_valores = []

for feature in ic_features_combined["feature"]:
    if feature in mnq_features_corr.columns:
        corrs = mnq_features_corr[feature].drop(labels=[feature])  # eliminar autocorrelación
        top5 = corrs.abs().sort_values(ascending=False).head(5)
        features = top5.index.tolist()
        valores = [mnq_features_corr.loc[factor, f] for f in features]
    else:
        features = [None, None, None]
        valores = [None, None, None]

    top5_features.append(features)
    top5_valores.append(valores)

# Paso 4: añadir al DataFrame original como columnas separadas
ic_features_combined["corr_ind_1"] = [f[0] for f in top5_features]
ic_features_combined["corr_value_1"] = [v[0] for v in top5_valores]

ic_features_combined["corr_ind_2"] = [f[1] for f in top5_features]
ic_features_combined["corr_value_2"] = [v[1] for v in top5_valores]

ic_features_combined["corr_ind_3"] = [f[2] for f in top5_features]
ic_features_combined["corr_value_3"] = [v[2] for v in top5_valores]


In [58]:
# Seleccionar solo las columnas deseadas
ic_top_corr= ic_features_combined[["feature", "corr_ind_1", "corr_ind_2", "corr_ind_3"]].copy()

### 3.4. Resultados

In [59]:
ic_top_corr

,feature,corr_ind_1,corr_ind_2,corr_ind_3
0,rsi_14,rsi_7,bb_30,stoch_k_30
1,rsi_7,bb_20,rsi_14,stoch_k_20
0,momentum_10,price_ema20,price_ema30,macd
1,momentum_5,price_ema20,momentum_10,price_ema30
0,macd,momentum_10,price_ema20,momentum_5
1,price_ema20,price_ema30,momentum_10,roc_20
2,price_ema30,price_ema20,roc_20,roc_30
1,stoch_k_20,bb_20,stoch_k_30,bb_30
2,stoch_k_30,bb_30,stoch_k_20,rsi_14
1,bb_20,stoch_k_20,rsi_7,bb_30


### 3.5. Análisis de resultados

1. Indicadores clásicos de momentum y osciladores

    - `rsi_14` está fuertemente correlacionado con `rsi_7`, `bb_30` y `stoch_k_30`.  
    - `rsi_7` se conecta con `bb_20`, `rsi_14`, `stoch_k_20`.  

    Esto confirma que los RSI están muy ligados a osciladores tipo stochastics y bandas de Bollinger. Si los uso juntos puedo redundar en información repetida.
<br>

2. Momentum lineal y promedios móviles

    - `momentum_10` está cercano a `price_ema20`, `price_ema30`, `macd`.  
    - `momentum_5` está ligado a `price_ema20`, `momentum_10`, `price_ema30`.  
    - `macd` está conectado a `momentum_10`, `price_ema20`, `momentum_5`.  
    - `price_ema20` y `price_ema30` están muy correlacionados entre sí y también con `momentum` y `roc`.  

    Esto dibuja un clúster de momentum tendencial, donde todos capturan la misma señal de continuidad de tendencia. Si incluyo demasiados de este grupo me pueden generar colinealidad.
<br>

3. Osciladores estocásticos y Bollinger

    - `stoch_k_20` ligado a `bb_20`, `stoch_k_30`, `bb_30`.  
    - `stoch_k_30` ligado a `bb_30`, `stoch_k_20`, `rsi_14`.  
    - `bb_20` ligado a `stoch_k_20`, `rsi_7`, `bb_30`.  
    - `bb_30` ligado a `rev_score_30`, `stoch_k_30`, `bb_20`.  
    - `bb_60` ligado a `rev_score_60`, `rev_score_45`, `rsi_14`.  

    Aquí se ve un clúster claro de osciladores-volatilidad. Están fuertemente interrelacionados y tienden a medir sobrecompra/sobreventa y rangos de precios. El riesgo de redundancia es alto.
<br>

4. Factores de reversión de precio

    - Los `rev_mom_z_*` se relacionan entre sí y también con `rev_score` y `rev_mom_vol_z`.  
      - Por ejemplo el `rev_mom_z_90` está correlacionado con `rev_mom_vol_z_90`, `rev_mom_z_60`, `rev_score_90`.  
    - Los `rev_score_*` forman un subgrupo muy cohesionado, relacionados con las Bollinger (`bb_30`, `bb_60`).  
    - Las `rev_mom_vol_z_*` están interconectados entre sí y con `rev_mom_z`.  

    Esto indica que la familia de reversal scores y momentum de reversión es muy densa en correlaciones internas, casi redundante. Probablemente baste con elegir un par representativos (como por ej. `rev_mom_z_90`, `rev_score_90`, `rev_mom_vol_z_60`).
<br>

5. Retornos intradía extremos (IRE)

    - `ire_60` ligado a `ire_90`, `rev_mom_z_90`, `rev_score_90`.  
    - `ire_90` ligado a `ire_60`, `rev_mom_z_90`, `rev_score_90`.  

    Este par es extremadamente fuerte y además conecta con los reversales. Los IRE aportan una visión macro de agotamiento intradía que está alineada con reversión.
<br>

6. Volatilidad

    - El `atr_norm` está correlacionado con `ire_90`, `ire_60`, `bb_60`.  

    El ATR queda como un puente entre volatilidad clásica y factores de reversión intradía.


### 3.6. Conclusiones de correlación

1. Hay clústers muy definidos:  
    - Momentum y medias móviles (`momentum`, `ema`, `macd`, `roc`).  
    - Osciladores y volatilidad (`rsi`, `stoch_k`, `bb`).  
    - Reversión (`rev_mom_z`, `rev_score`, `rev_mom_vol_z`).  
    - IRE (`ire_60`, `ire_90`).  

2. Dentro de cada clúster, los factores son altamente correlacionados, por lo que elegir demasiados genera redundancia.  

3. Una buena estrategia sería seleccionar 1 o 2 factores representativos por clúster:  
    - Momentum: `roc_60` y `price_ema30`.  
    - Osciladores: `bb_60` y `stoch_k_30`.  
    - Reversión: `rev_mom_z_90` y `rev_score_90`.  
    - IRE: `ire_60` o `ire_90`.  
    - Volatilidad: `atr_norm`.  

De esta forma, nuestro modelo tendría cobertura balanceada sin sobrecargarlo con señales repetitivas.

## 4. ¿Como hago una revisión cruzada?

In [61]:
ic_features_comparison

,features_to_30min,features_to_60min,features_to_90min
0,ire_90,ire_60,ire_60
1,ire_60,ire_90,ire_90
2,rev_mom_z_90,rev_mom_z_90,rev_mom_z_90
3,roc_60,roc_60,roc_60
4,rev_mom_z_60,rev_mom_z_60,rev_mom_z_60
5,rev_mom_vol_z_90,bb_60,bb_60
6,rev_score_90,rev_score_90,stoch_k_30
7,rev_mom_vol_z_60,rev_mom_vol_z_90,price_ema30
8,bb_60,rev_mom_z_45,rev_mom_z_45
9,rev_mom_z_45,stoch_k_30,rev_score_90


In [63]:
# Paso 1: lista de top N indicadores en cada horizonte
top_n = 8
top_30 = ic_features_comparison["features_to_30min"].head(top_n).tolist()
top_60 = ic_features_comparison["features_to_60min"].head(top_n).tolist()
top_90 = ic_features_comparison["features_to_90min"].head(top_n).tolist()

print("top features to 30min:", top_30)
print("top features to 60min:", top_60)
print("top features to 90min:", top_90)

# Paso 2: combinamos todos los top N
top_indicadores = set(top_30 + top_60 + top_90)

print("\nFeatures destacados en algún horizonte:", top_indicadores)



top features to 30min: ['ire_90', 'ire_60', 'rev_mom_z_90', 'roc_60', 'rev_mom_z_60', 'rev_mom_vol_z_90', 'rev_score_90', 'rev_mom_vol_z_60']
top features to 60min: ['ire_60', 'ire_90', 'rev_mom_z_90', 'roc_60', 'rev_mom_z_60', 'bb_60', 'rev_score_90', 'rev_mom_vol_z_90']
top features to 90min: ['ire_60', 'ire_90', 'rev_mom_z_90', 'roc_60', 'rev_mom_z_60', 'bb_60', 'stoch_k_30', 'price_ema30']

Features destacados en algún horizonte: {'ire_90', 'rev_mom_z_90', 'rev_mom_vol_z_90', 'rev_mom_z_60', 'rev_score_90', 'rev_mom_vol_z_60', 'roc_60', 'bb_60', 'price_ema30', 'stoch_k_30', 'ire_60'}


In [64]:
# Paso 3: filtrar correlaciones SOLO de los indicadores destacados
ic_corr_filt = ic_top_corr[ic_top_corr["feature"].isin(top_indicadores)]

print("\nCorrelaciones de los indicadores más relevantes:")
ic_corr_filt




Correlaciones de los indicadores más relevantes:


,feature,corr_ind_1,corr_ind_2,corr_ind_3
2,price_ema30,price_ema20,roc_20,roc_30
2,stoch_k_30,bb_30,stoch_k_20,rsi_14
3,bb_60,rev_score_60,rev_score_45,rsi_14
4,roc_60,rev_mom_z_60,roc_30,price_ema30
2,rev_mom_z_60,rev_mom_z_45,rev_mom_vol_z_60,roc_60
3,rev_mom_z_90,rev_mom_vol_z_90,rev_mom_z_60,rev_score_90
6,rev_score_90,rev_score_60,bb_60,rev_score_45
2,rev_mom_vol_z_60,rev_mom_vol_z_45,rev_mom_vol_z_90,rev_mom_z_60
3,rev_mom_vol_z_90,rev_mom_vol_z_60,rev_mom_vol_z_45,rev_mom_z_90
1,ire_60,ire_90,rev_mom_z_90,rev_score_90


In [65]:
# Paso 4: ver qué indicadores aparecen repetidamente como correlacionados
from collections import Counter

corrs = (
    ic_corr_filt[["corr_ind_1", "corr_ind_2", "corr_ind_3"]]
    .values.flatten()
)
conteo_corr = Counter(corrs)

print("\nIndicadores que más aparecen como correlacionados con los top:")
for ind, count in conteo_corr.most_common():
    print(f"{ind}: {count} veces")


Indicadores que más aparecen como correlacionados con los top:
rev_mom_z_60: 3 veces
rev_score_90: 3 veces
rev_mom_z_90: 3 veces
roc_30: 2 veces
rsi_14: 2 veces
rev_score_60: 2 veces
rev_score_45: 2 veces
rev_mom_vol_z_60: 2 veces
rev_mom_vol_z_90: 2 veces
rev_mom_vol_z_45: 2 veces
price_ema20: 1 veces
roc_20: 1 veces
bb_30: 1 veces
stoch_k_20: 1 veces
price_ema30: 1 veces
rev_mom_z_45: 1 veces
roc_60: 1 veces
bb_60: 1 veces
ire_90: 1 veces
ire_60: 1 veces


#HASTA ACÁ LLEGAMOS 20/09/2025

### 4.4. Selección de mejores indicadores técnicos

 Justificación de los parámetros elegidos para selección de mejores indicadores técnicos:
<br><br>

**Umbral de Information Coefficient: `umbral_ic = 0.4`**
- El IC_score mide la estabilidad y poder predictivo de un indicador respecto al retorno objetivo. En literatura cuantitativa, un `IC_score` ≥ 0.4 se considera sólido.
- Estos factores son consistentes en su señal predictiva y valen la pena incluirlos si no son redundantes. Al usar este valor como umbral principal, priorizás calidad estadística.
<br><br>

**Umbral flexible de Information Coefficient: ` ic_flexible` = 0.35**
- Este es un valor límite inferior para aceptar factores con señal más débil, solo si aportan diversidad. Factores con `IC_score` entre 0.35 y 0.4 no son necesariamente inútiles, pero requieren una segunda condición para ser aceptados.
- Este umbral permite flexibilidad para incluir señales complementarias poco correlacionadas con los factores fuertes. Esto ayuda a evitar perder factores que, si bien son débiles por sí solos, pueden mejorar el modelo al combinarse con otros.
<br><br>

**Umbral máximo de correlación:  `max_corr_con_seleccionados` < 0.925**
- Umbral general para evitar incluir factores altamente correlacionados.
- Correlaciones por encima de 0.925 suelen indicar redundancia significativa (misma señal expresada con distinto nombre). Este umbral mantiene un buen balance entre señal predictiva y diversidad, como observaste empíricamente en tus pruebas.
- Elegido como término medio entre la pérdida de información (0.95) y la fragmentación excesiva (0.90).
<br><br>

**Umbral flexible de correlación: `umbral_corr_flexible` = 0.7**
- Es el umbral de correlación máxima permitida entre un factor débil (`IC_score` ≥ 0.35) y los ya seleccionados.
- Se basa en la idea de diversidad de señales: aceptamos factores "débiles" solo si aportan información nueva. Una correlación < 0.7 implica que el nuevo factor no es muy similar a ninguno de los ya elegidos.
- Este filtro mitiga el riesgo de redundancia y overfitting.


**Conclusión:**

Esta lógica de selección mixta es sólida porque:

- Prioriza factores con buen IC.
- Permite flexibilidad controlada para incluir factores útiles pero menos fuertes.
- Favorece la diversidad de señales al controlar la colinealidad.


In [76]:
ic_features_combined

,feature,ic_mean_30min,ic_std_30min,ic_mean_60min,ic_std_60min,ic_mean_90min,ic_std_90min,ic_score_30min,ic_score_60min,ic_score_90min,corr_ind_1,corr_value_1,corr_ind_2,corr_value_2,corr_ind_3,corr_value_3
0,rsi_14,-0.088180,0.172203,-0.132798,0.200280,-0.176524,0.202798,0.512072,0.663061,0.870443,rsi_7,0.936097,bb_30,0.935810,stoch_k_30,0.925944
1,rsi_7,-0.062888,0.137946,-0.095645,0.152071,-0.127395,0.152254,0.455891,0.628948,0.836727,bb_20,0.945422,rsi_14,0.936097,stoch_k_20,0.921289
0,momentum_10,-0.053240,0.129744,-0.082424,0.136893,-0.112294,0.134865,0.410347,0.602103,0.832637,price_ema20,0.895579,price_ema30,0.847844,macd,0.824105
1,momentum_5,-0.040213,0.093976,-0.060545,0.097287,-0.082049,0.096481,0.427903,0.622331,0.850414,price_ema20,0.791655,momentum_10,0.707036,price_ema30,0.702394
0,macd,-0.018010,0.135153,-0.026394,0.122985,-0.035572,0.121715,0.133256,0.214615,0.292255,momentum_10,0.824105,price_ema20,0.710716,momentum_5,0.665269
1,price_ema20,-0.072318,0.158994,-0.111052,0.177833,-0.151308,0.174142,0.454850,0.624473,0.868880,price_ema30,0.980876,momentum_10,0.895579,roc_20,0.863443
2,price_ema30,-0.087274,0.176637,-0.132857,0.203450,-0.180860,0.199570,0.494089,0.653024,0.906247,price_ema20,0.980876,roc_20,0.901228,roc_30,0.863859
1,stoch_k_20,-0.070241,0.147213,-0.105944,0.159769,-0.141962,0.160943,0.477135,0.663105,0.882061,bb_20,0.948167,stoch_k_30,0.928436,bb_30,0.922871
2,stoch_k_30,-0.087019,0.170373,-0.132252,0.192868,-0.175458,0.193385,0.510757,0.685712,0.907298,bb_30,0.946797,stoch_k_20,0.928436,rsi_14,0.925944
1,bb_20,-0.063539,0.141046,-0.097252,0.152624,-0.130950,0.154313,0.450485,0.637197,0.848595,stoch_k_20,0.948167,rsi_7,0.945422,bb_30,0.941975


In [77]:
# Parámetros de selección
#umbral_ic = 0.4
#ic_flexible = 0.35
#umbral_corr_flexible = 0.7

def features_selection(ventana: str, umbral_ic = 0.4, ic_flexible = 0.5, umbral_corr_flexible = 0.7):
  # Inicializar lista de factores seleccionados
  mejores_indicadores_tecnicos = []

  # Ordenar IC_indicadores por IC_score descendente
  df_ordenado = ic_features_combined.sort_values(by=f"ic_score_{ventana}min", ascending=False).reset_index(drop=True)

  # Iterar por cada fila
  for i, fila in df_ordenado.iterrows():
      factor = fila["feature"]
      ic_score = fila[f"ic_score_{ventana}min"]

      # Obtener lista de los 5 factores más correlacionados
      correlaciones = [
          fila.get(f"corr_value_{i}", 0) for i in range(1, 6)  #corr_ind_1	corr_value_1
          if fila.get(f"corr_ind_{i}") in mejores_indicadores_tecnicos
      ]
      max_corr_con_seleccionados = max([abs(c) for c in correlaciones], default=0)

      # Criterios de selección
      if ic_score >= umbral_ic:
          if max_corr_con_seleccionados < 0.925:  # opcional: no meter duplicados
              mejores_indicadores_tecnicos.append(factor)
      elif ic_score >= ic_flexible and max_corr_con_seleccionados < umbral_corr_flexible:
          mejores_indicadores_tecnicos.append(factor)
  return mejores_indicadores_tecnicos


In [78]:
best_features_30 = features_selection(ventana="30")
best_features_60 = features_selection(ventana="60")
best_features_90 = features_selection(ventana="90")

In [79]:
best_features_30

['ire_90',
 'rev_mom_z_90',
 'roc_60',
 'rev_mom_z_60',
 'rev_mom_vol_z_90',
 'rev_score_90',
 'rev_mom_vol_z_60',
 'bb_60',
 'rev_mom_z_45',
 'rsi_14',
 'rev_mom_vol_z_45',
 'price_ema30',
 'stoch_k_20',
 'bb_30',
 'rev_score_45',
 'roc_30',
 'rev_mom_z_30',
 'momentum_5',
 'roc_20',
 'momentum_10']

In [80]:
best_features_60

['ire_60',
 'rev_mom_z_90',
 'roc_60',
 'rev_mom_z_60',
 'bb_60',
 'rev_score_90',
 'rev_mom_vol_z_90',
 'rev_mom_z_45',
 'stoch_k_30',
 'rev_mom_vol_z_60',
 'price_ema30',
 'rev_mom_vol_z_45',
 'bb_20',
 'roc_30',
 'rev_mom_z_30',
 'momentum_5',
 'rev_score_45',
 'roc_20',
 'momentum_10']

Las recomendaciones finales de indicadores técnicos son:

**RSI (Relative Strength Index):**

- `rsi_14` debe ser priorizado como alpha factor principal dentro de este grupo. Su `IC_score` es el más alto entre todos los indicadores evaluados, con excelente estabilidad y baja redundancia. Representa una señal sólida de sobrecompra/sobreventa con una ventana estándar.
- `rsi_7` puede considerarse como alternativa secundaria, ofreciendo una versión más reactiva del RSI que captura mejor las señales de corto plazo.
-`rsi_3` puede considerarse como un complemento táctico, útil en estrategias intradía con horizontes de muy corto plazo, aunque su señal puede ser más ruidosa.

**Medias móviles:**
- `price_ema30` se recomienda como medida de tendencia suave, útil para detectar desviaciones del precio respecto a su nivel de equilibrio de corto-mediano plazo.
- Este factor no está altamente correlacionado con el resto de los seleccionados, y aporta contexto de fondo ideal para estrategias de reversión.

**Estocásticos:**
- `stoch_k_20` fue seleccionado como el más robusto dentro del grupo, combinando buen `IC_score` con menor redundancia.
- Este indicador captura zonas de sobrecompra/sobreventa de manera más sensible que el RSI, y es ideal para escenarios de lateralización o saturación de precios.


**Bollinger Bands (%B):**
- `bb_percent_30_20` es la variante más recomendable, gracias a su buen `IC_score` y capacidad para anticipar reversiones dentro de rangos amplios.
- `bb_percent_20_15` también es una alternativa válida, especialmente si se busca una señal más rápida con un rango de observación más corto.
- No es necesario conservar todas las combinaciones posibles. Es suficiente incluir una o dos versiones representativas con ventanas de 20 y 30, dado que las demás son altamente correlacionadas entre sí.
- Este grupo es fundamental para detectar extremos de precio y posibles rompimientos de rango.

**Momentum y Rate of Change (ROC):**
- `momentum_3`, `momentum_10`, `roc_5` y `roc_20` conforman un conjunto de señales complementarias que capturan el impulso del mercado en distintos horizontes.
- `momentum_3` y `roc_5` son útiles para señales rápidas.
- `momentum_10` y `roc_20` ofrecen contexto de mediano plazo.
- La inclusión de varios horizontes permite al modelo adaptarse a distintos regímenes de velocidad del mercado.

**Volatilidad (ATR):**

- `atr_norm` se destaca como uno de los alpha factors más valiosos, a pesar de tener un IC_score algo por debajo del umbral estricto.
- Su baja colinealidad con otros factores y su naturaleza distinta (captura de volatilidad en lugar de dirección) justifican plenamente su inclusión.
- Es un factor complementario clave para adaptar estrategias a contextos de alta o baja volatilidad intradía.



### 4.5. Función para añadir indicadores técnicos seleccionados


['rsi_14', 'price_ema30', 'stoch_k_20', 'bb_percent_30_20', 'rsi_7', 'rsi_3', 'bb_percent_20_15', 'momentum_3', 'roc_5', 'roc_20', 'momentum_10', 'atr_norm']


In [ ]:
mejores_indicadores_tecnicos = [
    'momentum_3',
    'momentum_10',
    'roc_5',
    'roc_20',
    'rsi_3',
    'rsi_7',
    'rsi_14',
    'stoch_k_20',
    'bb_percent_20_15',
    'bb_percent_30_20',
    'price_ema30',
    'atr_norm']




In [ ]:
def best_indicators(df=mnq_intraday, target='close' ):

  def aplicar_por_dia (grupo):
        grupo = grupo.copy()

        # MOMENTUM y VELOCIDAD
        grupo['momentum_3'] = grupo[target].pct_change(3)
        grupo['momentum_10'] = grupo[target].pct_change(10)
        grupo['roc_5'] = ROCIndicator(close=grupo[target], window=5).roc()
        grupo['roc_20'] = ROCIndicator(close=grupo[target], window=20).roc()

        # SOBRECOMPRA / SOBREVENTA (RSI)
        grupo['rsi_3'] = ta.momentum.RSIIndicator(grupo[target], window=3).rsi()
        grupo['rsi_7'] = ta.momentum.RSIIndicator(grupo[target], window=7).rsi()
        grupo['rsi_14'] = ta.momentum.RSIIndicator(grupo[target], window=14).rsi()

        stoch_20 = StochasticOscillator(high=grupo['high'], low=grupo['low'], close=grupo[target], window=20, smooth_window=3)
        grupo['stoch_k_20'] = stoch_20.stoch()

        # BOLLINGER BANDS
        grupo['bb_percent_20_15'] = BollingerBands(grupo[target], window=20, window_dev=1.5).bollinger_pband()
        grupo['bb_percent_30_20'] = BollingerBands(grupo[target], window=30, window_dev=2).bollinger_pband()

        # TENDENCIA RELATIVA
        grupo['price_ema30'] = grupo[target] / grupo[target].ewm(span=30).mean() - 1

        # VOLATILIDAD
        atr = AverageTrueRange(high=grupo['high'], low=grupo['low'], close=grupo[target], window=14)
        grupo['atr'] = atr.average_true_range()
        grupo['atr_norm'] = grupo['atr'] / grupo[target]
        grupo.drop(columns=['atr'], inplace=True)  #Elimino columna 'atr'
        return grupo

  df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)
  return df

In [ ]:
mnq_intraday = best_indicators()

In [ ]:
mnq_intraday

## 5. Análisis del IC durante el día.

### 5.1. Funciones

#### Función para plotear el comportamiento

In [ ]:
def plot_ic_tecnicos_por_minuto(df: pd.DataFrame,
                                 factors: list[str],
                                 target: str,
                                 date_col: str = 'date'):
    """
    Calcula y grafica el IC por minuto intradía para una lista de indicadores técnicos.
    El eje X muestra el tiempo intradía (desde las 08:00 AM).
    """

    df = df.copy()
    df['minute_index'] = df.groupby(date_col).cumcount()

    def compute_ic_series(factor_col, target_col):
        grouped = df[[factor_col, target_col, 'minute_index']].dropna()
        return grouped.groupby('minute_index').apply(
            lambda x: x[factor_col].corr(x[target_col])
        )

    # Calcular IC por minuto para cada factor
    ic_series = {factor: compute_ic_series(factor, target) for factor in factors}

    # Eje X: minutos del día y etiquetas horarias
    xticks = np.arange(0, 451, 30)
    start_time = pd.to_datetime("08:00")
    time_labels = [(start_time + pd.Timedelta(minutes=i)).strftime("%H:%M") for i in xticks]

    # Graficar
    plt.figure(figsize=(15, 6))
    for factor, ic in ic_series.items():
        plt.plot(ic.index, ic, label=factor)

    plt.axhline(0, color='black', linestyle='--', linewidth=1)
    plt.title(f'IC por minuto del día - Target: {target}')
    plt.xlabel('Hora intradía')
    plt.ylabel('Information Coefficient (IC)')
    plt.xticks(ticks=xticks, labels=time_labels, rotation=45)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    return ic_series

#### Función para analizar los mejores y peores IC durante el día

In [ ]:
def analyze_intraday_ic_extremes(ic_minute_df: pd.DataFrame,
                                 mode: str = 'worst',
                                 top_n: int = 30,
                                 market_open: str = "08:00") -> pd.DataFrame:
    """
    Analiza series de IC minuto a minuto para distintos alpha factors.
    Retorna el IC mínimo, IC máximo, y el promedio de los top_n peores o mejores ICs,
    indicando la hora exacta en la que ocurren los extremos.

    Parámetros:
    - ic_minute_df: DataFrame con columnas = alpha factors, índice = minuto (0 a 450)
    - mode: 'worst' para los peores ICs, 'best' para los mejores ICs
    - top_n: cantidad de valores extremos a promediar
    - market_open: hora de inicio del mercado en formato "HH:MM" (por defecto 08:30)

    Retorna:
    - DataFrame resumen ordenado por el promedio de los ICs extremos
    """
    results = []
    base_time = pd.to_datetime(market_open)

    for col in ic_minute_df.columns:
        series = ic_minute_df[col].dropna()

        if mode == 'worst':
            extreme_vals = series.nsmallest(top_n)
        elif mode == 'best':
            extreme_vals = series.nlargest(top_n)
        else:
            raise ValueError("El parámetro 'mode' debe ser 'worst' o 'best'.")

        extreme_mean = extreme_vals.mean()

        # IC mínimo y su hora
        min_ic = series.min()
        min_minute = series.idxmin()
        min_time = (base_time + pd.Timedelta(minutes=min_minute)).strftime("%H:%M")

        # IC máximo y su hora
        max_ic = series.max()
        max_minute = series.idxmax()
        max_time = (base_time + pd.Timedelta(minutes=max_minute)).strftime("%H:%M")

        results.append({
            'Alpha Factor': col,
            'IC mínimo': min_ic,
            'Hora del IC mínimo': min_time,
            'IC máximo': max_ic,
            'Hora del IC máximo': max_time,
            f'Promedio de los {top_n} {"peores" if mode == "worst" else "mejores"} ICs': extreme_mean
        })

    result_df = pd.DataFrame(results).set_index('Alpha Factor')
    sort_col = result_df.columns[-1]
    result_df = result_df.sort_values(sort_col, ascending=(mode == 'worst'))

    return result_df

#### Función para analizar el IC por ventana de tiempo.

In [ ]:
def analizar_ic_ventanas_min(ic_por_minuto: dict[str, pd.Series], minutos_inicio=480, ventana=30):
    """
    Evalúa los bloques de ventana minutos consecutivos con peor y mejor IC promedio para cada factor.
    Devuelve una tabla con métricas resumidas.
    """
    resultados = []

    def minuto_a_hora(minuto):
        total_min = minutos_inicio + minuto
        hora = pd.Timestamp("00:00") + pd.Timedelta(minutes=total_min)
        return hora.strftime("%H:%M")

    for factor, serie in ic_por_minuto.items():
        serie = serie.dropna()

        # Calcular promedio móvil de 30 minutos
        promedio_ventana = serie.rolling(window=ventana, min_periods=ventana).mean()

        # Encontrar ventana con mejor y peor promedio
        peor_inicio = promedio_ventana.idxmin()
        mejor_inicio = promedio_ventana.idxmax()

        peor_valor = promedio_ventana.loc[peor_inicio]
        mejor_valor = promedio_ventana.loc[mejor_inicio]

        rango_peor = f"{minuto_a_hora(peor_inicio)} - {minuto_a_hora(peor_inicio + ventana - 1)}"
        rango_mejor = f"{minuto_a_hora(mejor_inicio)} - {minuto_a_hora(mejor_inicio + ventana - 1)}"

        resultados.append({
            "Alpha Factor": factor,
            f"Promedio peor bloque {ventana}min": peor_valor,
            f"Rango peor bloque {ventana}min": rango_peor,
            f"Promedio mejor bloque {ventana}min": mejor_valor,
            f"Rango mejor bloque {ventana}min": rango_mejor
        })

    return pd.DataFrame(resultados).sort_values(f"Promedio mejor bloque {ventana}min", ascending=False).reset_index(drop=True)

### 5.2. Gráfico de comportamiento

In [ ]:
# 1. Calcular el IC por minuto para los mejores factores
ic_por_minuto = plot_ic_tecnicos_por_minuto(mnq_intraday, mejores_indicadores_tecnicos, target='target_return_30')

Observemos el comportamiento de los IC en ventanas de 60 min, y busquemos cual es la mejor y la peor hora para tomar en cuenta estos indicadores.

In [ ]:
ic_minute_df = pd.DataFrame(ic_por_minuto)
resumen_60 = analizar_ic_ventanas_min(ic_minute_df,  ventana=120)

In [ ]:
resumen_60

Con base en el análisis sobre el IC promedio por bloques de 60 minutos, puedo extraer varias conclusiones relevantes para la operativa intradía y la selección temporal de tus indicadores técnicos:

**1. El mejor momento para operar con indicadores técnicos es entre las 13:39 y 14:49**
- La gran mayoría de los factores muestran su mejor rendimiento predictivo en este rango horario: `price_ema30`, `roc_20`, `rsi_14`, `bb_percent_30_20`, `bb_percent_20_15`, `stoch_k_20`, `momentum_3`, `rsi_7`, `rsi_3`...
- Esto sugiere que hay más consistencia y direccionalidad en el mercado durante este tramo, probablemente debido a una menor presencia de “ruido” y una mayor participación institucional pre-cierre.

**2. El peor rendimiento se concentra entre las 09:41 y 11:34**
- Indicadores como `atr_norm`, `price_ema30`, `roc_20`, `momentum_10`, `roc_5`, `rsi_14`, `bb_percent_30_20` tienen sus bloques más débiles en esta franja.
- Esto podría estar relacionado con la alta volatilidad y comportamiento errático de la apertura, que puede confundir a muchos factores técnicos.

**3. Algunos indicadores son más estables y consistentes**
- `atr_norm` y `rsi_14` tienen promedios negativos muy bajos en sus peores bloques, lo que indica que incluso en sus peores momentos, no generan señales fuertemente equivocadas.
- Esto los convierte en candidatos robustos para modelos que busquen consistencia a lo largo del día.

**4. Otros factores son altamente sensibles al contexto horario**
- `price_ema30` y `roc_20` tienen diferencias marcadas entre su peor y mejor bloque (más de 0.13 en IC), mostrando que su utilidad depende mucho del horario en que se usan.
- Podrías considerar usar estos factores solo en rangos temporales favorables o ponderarlos según la hora del día en tu modelo predictivo.

**Recomendaciones prácticas**

- Para entrenamiento de modelos o ejecución de señales: prioriza el uso de datos entre las 13:39 y 14:49.
- Si tu modelo es sensible a errores, evita tomar decisiones automáticas entre las 10:00 y 11:30, o usa estrategias conservadoras en ese rango.
- Considera agregar la hora del día como variable explicativa, o entrenar modelos separados por bloques horarios (por ejemplo: apertura, mediodía, pre-cierre).




In [ ]:
mnq_intraday